# 06 - حل المشكلات المعقدة في الـ PDF وتنظيف النصوص (Handling Common PDF Issues)

في هذا الدفتر ننتقل إلى المعمارية المعيارية المتقدمة لمعالجة ملفات الـ PDF:
1. **تنظيف النصوص**: إزالة المسافات المتعددة والأسطر الفارغة الزائدة والرموز المشفرة (Ligatures).
2. **بناء كلاس `SmartPDFProcessor`**: معمارية نظيفة (Modular Class) تدمج التحميل، والتنظيف، والتجزئة، وتخطي الصفحات الفارغة.
3. **إثراء الميتا-داتا (Metadata Enrichment)**: تزويد كل قطعة نصية برقم الصفحة الفعلي، وإجمالي الصفحات، وعدد الأحرف، وطريقة التجزئة.

### الخطوة 1: تجربة دالة تنظيف النصوص البسيطة (`clean_text`)

نص خام يحمل تشوهات شائعة في ملفات الـ PDF (مسافات متباعدة، أسطر فارغة مكررة، أحرف مدمجة).

In [ ]:
def clean_text(text: str) -> str:
    # 1. استبدال الـ Ligatures الشائعة
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    # 2. توحيد المسافات والأسطر الفارغة المكررة
    text = " ".join(text.split())
    return text.strip()

# نص تجريبي مشوه كما يخرج أحياناً من محركات استخراج الـ PDF
raw_pdf_text = """Company Financial Report 2024\n\n\n
The financial   performance for   fiscal year 2024 shows significant   growth in proﬁtability.\n\n
Revenue increased by 25% due to efﬁcient operations."""

cleaned = clean_text(raw_pdf_text)

print("--- النص الخام قبل التنظيف ---")
print(repr(raw_pdf_text))

print("\n--- النص بعد التنظيف ---")
print(cleaned)

### الخطوة 2: بناء كلاس المعالجة المتقدم `SmartPDFProcessor`

معمارية متكاملة تقوم بالآتي:
* قراءة الملف صفحة بصفحة.
* تنظيف محتوى كل صفحة.
* فحص طول الصفحة وتخطي الصفحات الفارغة (`len < 50`).
* تجزئة النصوص وإثراء الـ Metadata بالمعلومات الحيوية.

In [ ]:
import os
from typing import List
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

class SmartPDFProcessor:
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

    def _clean_text(self, text: str) -> str:
        # استبدال الحروف المشفرة وتنظيف المسافات
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        return " ".join(text.split()).strip()

    def process_pdf(self, pdf_path: str) -> List[Document]:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"الملف غير موجود: {pdf_path}")

        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        total_pages = len(pages)
        processed_chunks: List[Document] = []

        for page_num, page in enumerate(pages, 1):
            clean_content = self._clean_text(page.page_content)

            # تخطي الصفحات شبه الفارغة (أقل من 50 حرف)
            if len(clean_content) < 50:
                print(f"⚠️ تم تخطي الصفحة {page_num} لقلة المحتوى ({len(clean_content)} حرف).")
                continue

            # إثراء الميتا-داتا
            enhanced_metadata = {
                **page.metadata,
                "page_number": page_num,
                "total_pages": total_pages,
                "chunk_method": "smart_recursive",
                "char_count": len(clean_content)
            }

            # إنشاء قطع Document مجزأة مع الميتا-داتا الجديدة
            chunks = self.text_splitter.create_documents(
                texts=[clean_content],
                metadatas=[enhanced_metadata]
            )
            processed_chunks.extend(chunks)

        return processed_chunks

print("✅ تم تعريف كلاس SmartPDFProcessor بنجاح!")

### الخطوة 3: تشغيل المعالج على ملف الـ PDF المخصص (`sample_financial_report.pdf`)

الملف يحتوي على:
* صفحة 1: تقرير مالي بمسافات وأسطر زائدة.
* صفحة 2: صفحة شبه فارغة (`[Blank Page]`) لاختبار التخطي التلقائي.
* صفحة 3: توقعات مستقبلية وتفاصيل إضافية.

In [ ]:
# اختبار الكلاس الذكي على التقرير المالي التجريبي
from pathlib import Path
data_dir = Path('data') if Path('data').exists() else Path('../../data')
pdf_file = str((data_dir / 'pdf' / 'sample_financial_report.pdf').resolve())

processor = SmartPDFProcessor(chunk_size=300, chunk_overlap=40)
chunks = processor.process_pdf(pdf_file)

print(f"[+] إجمالي القطع الناتجة بعد التنظيف والفلترة: {len(chunks)}\n")
for i, chunk in enumerate(chunks, 1):
    print(f"--- القطعة #{i} (صفحة {chunk.metadata['page_number']}) ---")
    print(chunk.page_content[:180] + '...')
    print(f"الميتا-داتا: {chunk.metadata}\n")


### الخطوة 4: فحص الميتا-داتا المثرية والقطع المستخرجة

In [ ]:
for i, chunk in enumerate(smart_chunks, 1):
    print(f"=== Chunk {i} ===")
    print(f"المحتوى: {chunk.page_content}")
    print("\nالبيانات الوصفية المثرية (Enriched Metadata):")
    for k, v in chunk.metadata.items():
        print(f"  • {k}: {v}")
    print("=" * 50)

### خلاصة الدرس والانتقال للمرحلة التالية:

1. **تنظيف النصوص المسبق** يمنع تلوث الـ Embeddings بالمسافات والأحرف المشفرة.
2. **تخطي الصفحات الفارغة** يوفر تكاليف الحوسبة ومساحة التخزين في الـ Vector DB.
3. **إثراء الميتا-داتا** (`page_number`, `total_pages`, `char_count`) يجعل محرك البحث قادراً على تصفية وتوثيق كل فقرة بدقة.

المحاضرة التالية: **`07-word-documents`** (قراءة وتحليل مستندات Word بصيغة DOCX).